<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 7 章练习解答

## 练习 7.1：更改 prompt 风格

**练习目的**：将 Alpaca 风格的 prompt 模板替换为 Phi-3 风格，观察对训练和生成的影响。

假设我们有以下数据条目：

```json
{
  "instruction": "Identify the correct spelling of the following word.",
  "input": "Ocassion",
  "output": "The correct spelling is 'Occasion.'"
}
```

在主章节中，我们使用 Alpaca 风格的 prompt 模板进行格式化：

```
Below is an instruction that describes a task...
### Instruction:
### Input:
### Response:
```

在本练习中，我们使用 Phi-3 prompt 模板代替，格式如下：

```
<|user|>
Identify the correct spelling...
<|assistant|>
The correct spelling is 'Occasion'.
```

注意，这个 prompt 模板明显更短，这减少了微调和生成时的运行时间和硬件需求。要实现这个变更，我们更新 `format_input` 函数：

In [1]:
def format_input(entry):
    instruction_text = (
        f"<|user|>\n{entry['instruction']}"
    )

    input_text = f"\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

让我们通过将其应用到两个输入样本（一个有 input 一个没有）来验证它是否按预期工作：

In [2]:
sample_data = [
    {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}, 
    {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}
]

print(format_input(sample_data[0]))
print()
print(format_input(sample_data[1]))

<|user|>
Identify the correct spelling of the following word.
Ocassion

<|user|>
What is an antonym of 'complicated'?


接下来，我们还需要更新 `InstructionDataset` 类，使用 `<|assistant|>` prompt 模板来格式化响应：

```python
import tiktoken
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            # 使用 format_input 和 <|assistant|> 模板
            instruction_plus_input = format_input(entry)
            response_text = f"\n<|assistant|>:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))
    def __getitem__(self, index):
        return self.encoded_texts[index]
    def __len__(self):
        return len(self.data)

tokenizer = tiktoken.get_encoding("gpt2")
```

最后，我们还需要更新提取生成响应的方式：

```python
for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input(entry)
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    # 新: 调整 ###Response -> <|assistant|>
    response_text = generated_text[len(input_text):].replace("<|assistant|>:", "").strip()
    test_data[i]["model_response"] = response_text
```

为方便起见，练习解答实现在 [exercise_experiments.py](exercise_experiments.py) 脚本中，可以这样运行：

```bash
python exercise_experiments.py --exercise_solution phi3_prompt
```

在 Nvidia L4 GPU 上，使用 Phi-3 prompt 模板需要 1.5 分钟，而 Alpaca 模板需要 1.80 分钟。因此 Phi-3 模板大约快 17%，因为它的输入更短。

评估结果：Phi-3 prompt 平均得分 48.87，与 Alpaca 风格的得分在同一水平。

> **结论**：Phi-3 prompt 模板没有固有优势，但更简洁高效。注意特殊 token（如 `<|user|>`）在 GPT-2 tokenizer 中可能被拆分为多个 token ID，效率较低。

#### 提示：考虑特殊 token

- 注意，Phi-3 prompt 模板包含特殊 token，如 `<|user|>` 和 `<|assistant|>`，这对 GPT-2 tokenizer 来说可能不太理想
- GPT-2 tokenizer 只识别 `<|endoftext|>` 作为特殊 token（编码为 token ID 50256），其他特殊 token 会被拆分
- 例如，`<|user|>` 会被编码为 5 个独立的 token ID（27, 91, 7220, 91, 29），非常低效
- 如果你对如何扩展 tokenizer 和 LLM 来处理特殊 token 感兴趣，请参见 [extend-tiktoken.ipynb](../../ch05/09_extending-tokenizers/extend-tiktoken.ipynb)

&nbsp;
## 练习 7.2：指令和输入掩码

**练习目的**：在训练时将指令部分掩码（mask），只对响应部分计算 loss。

如下图所示，我们需要对 `InstructionDataset` 类和 `custom_collate_fn` 进行小幅修改。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/mask-instructions.webp" width=600px>

**图示：指令掩码策略**

```
原始 target:  [Below  is  an  instruction  ...  ### Response:  Crocodile  ...  <endoftext>]
掩码后:     [-100  -100  -100  -100      ...  ### Response:  Crocodile  ...  -100        ]
               ↑___ 指令部分被 -100 掩码 ___↑   ↑___ 只学习响应部分 ___↑  ↑ padding 也被掩码
```

> **实验结果**：指令掩码后平均得分 47.73，略低于不掩码的基线。与论文 "Instruction Tuning With Loss Over Instructions" 的发现一致。

In [4]:
# This `format_input` function is copied from the original chapter 7 code

def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

我们可以修改 `InstructionDataset` 类来收集指令的长度，然后在 collate 函数中使用这些长度来定位目标中的指令内容位置：

In [5]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        ##########################################################################################
        # New: Separate list for instruction lengths
        self.instruction_lengths = []
        ##########################################################################################
        
        self.encoded_texts = []
        
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

            ##########################################################################################
            # New: collect instruction lengths
            instruction_length = len(tokenizer.encode(instruction_plus_input))
            self.instruction_lengths.append(instruction_length)
            ##########################################################################################
            
    def __getitem__(self, index):
        # New: return both instruction lengths and texts separately
        return self.instruction_lengths[index], self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [6]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

接下来，我们更新 `custom_collate_fn`，其中每个 `batch` 现在是一个元组 `(instruction_length, item)`。此外，我们现在在目标 ID 列表中掩码相应的指令 token：

In [7]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # Find the longest sequence in the batch
    batch_max_length = max(len(item)+1 for instruction_length, item in batch)   # New: batch is now a tuple

    # Pad and prepare inputs and targets
    inputs_lst, targets_lst = [], []

    for instruction_length, item in batch:  # New: batch is now a tuple
        new_item = item.copy()
        # Add an <|endoftext|> token
        new_item += [pad_token_id]
        # Pad sequences to max_length
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets

        # Replace all but the first padding tokens in targets by ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        ##########################################################################################
        # New: Mask all input and instruction tokens in the targets
        targets[:instruction_length-1] = -100
        ##########################################################################################
        
        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]
        
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert list of inputs and targets to tensors and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

让我们在一些示例数据上试一试：

In [8]:
sample_data = [
    {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."},
    {'instruction': 'Sort the following list in alphabetical order.', 'input': 'Zebra, Elephant, Crocodile', 'output': 'Crocodile, Elephant, Zebra'},
    {'instruction': 'Arrange the given numbers in descending order.', 'input': '5, 12, 8, 3, 15', 'output': '15, 12, 8, 5, 3.'}
]

In [9]:
from torch.utils.data import DataLoader

train_dataset = InstructionDataset(sample_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=len(sample_data),
    collate_fn=custom_collate_fn,
    num_workers=0
)

In [10]:
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

Train loader:
torch.Size([3, 64]) torch.Size([3, 64])


In [11]:
print("Inputs:\n", inputs[1])
print("\n\nTargets:\n", targets[1])

Inputs:
 tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
          257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
        21017, 46486,    25,   198, 42758,   262,  1708,  1351,   287, 24830,
          605,  1502,    13,   198,   198, 21017, 23412,    25,   198,    57,
        37052,    11, 42651,    11,  9325, 19815,   576,   198,   198, 21017,
        18261,    25,   198,    34, 12204,   375,   576,    11, 42651,    11,
         1168, 37052, 50256, 50256])


Targets:
 tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,   198,   198, 21017, 18261,
           25,   198,    34, 12204,   375,   576,    11, 42651,    11,  1168,
      

如枑据 `targets` 张量所示，指令和填充 token 都已被 -100 掩码。让我们解码输入以确保它们看起来是正确的：

In [12]:
print(tokenizer.decode(list(inputs[1])))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Sort the following list in alphabetical order.

### Input:
Zebra, Elephant, Crocodile

### Response:
Crocodile, Elephant, Zebra<|endoftext|><|endoftext|>


接下来，让我们解码未被掩码的目标 token ID：

In [13]:
non_masked_targets = targets[1][targets[1] != -100]

print(tokenizer.decode(list(non_masked_targets)))



### Response:
Crocodile, Elephant, Zebra<|endoftext|>


如上所示，未被掩码的目标 token 排除了 "Instruction" 和 "Input" 字段，正如预期。现在可以运行修改后的代码来看模型使用这种掩码策略的效果。

```bash
python exercise_experiments.py --exercise_solution mask_instructions
```

评估结果：平均得分 47.73，略低于不掩码的基线。

&nbsp;
## 练习 7.3：在原始 Alpaca 数据集上微调

**练习目的**：使用原始 Stanford Alpaca 数据集（52k 条目，比主章节多 50 倍）进行微调。

只需将文件 URL 从：

```python
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json"
```

改为：

```python
url = "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json"
```

注意该数据集包含 52k 条目，建议在 GPU 上运行。如果遇到 OOM，可减小 batch size 或降低 `allowed_max_length`。

为方便起见，可以使用以下命令（batch_size=4, allowed_max_length=512）：

```bash
python exercise_experiments.py --exercise_solution alpaca_52k
```

训练完成后评估得分：48.16，略低于主章节。但 Alpaca 测试集包含更多样和更难的指令。

以下是 Alpaca 数据集中的一些示例，包括生成的模型响应：

```json
[
    {
        "instruction": "Edit the following sentence to increase readability: \"He made a huge effort and was so successful.\"",
        "input": "",
        "output": "He exerted a tremendous effort, and thus enjoyed great success.",
        "model_response": "He put in an immense effort and was rewarded with success."
    },
    ...
]
```

最后，使用 [ollama_evaluate.py](ollama_evaluate.py) 评估微调后的 LLM：

```bash
python ollama_evaluate.py --file_path instruction-data-with-response-alpaca52k.json
```

```
Average score: 48.16
```

得分略低于主章节使用的数据集。但注意 Alpaca 测试集包含更多样和更难的指令。

## 练习 7.4：使用 LoRA 进行参数高效微调

**练习目的**：使用 LoRA (Low-Rank Adaptation) 进行参数高效微调，只训练少量参数。

```
参数对比:
  全量微调: 406,286,336 (406M)
  LoRA 微调:   7,898,384 (7.9M)  ← 只有 1.9%
  → LoRA 快 28% (1.30 min vs 1.80 min)
  → 得分提升到 50.23
```

要使用 LoRA 进行指令微调，使用附录 E 中的相关类和函数：

```python
from appendix_E import LoRALayer, LinearWithLoRA, replace_linear_with_lora
```

然后，在 7.5 节的模型加载代码下方添加以下代码：

```python
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters before: {total_params:,}")

for param in model.parameters():
    param.requires_grad = False

replace_linear_with_lora(model, rank=16, alpha=16)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable LoRA parameters: {total_params:,}")
model.to(device)
```

为方便起见，可以使用以下命令（rank=16, alpha=16）：

```bash
python exercise_experiments.py --exercise_solution lora
```

在 Nvidia L4 GPU 上，LoRA 需 1.30 分钟，基线需 1.80 分钟。LoRA 快约 28%。

评估得分：50.23，与原始模型在同一水平。

> **总结**：LoRA 只训练 1.9% 的参数，速度快 28%，性能不降反而略升。是生产环境坦模型微调的首选方案。